# 03. Full fine-tuning of CamemBERT

The optional baseline of the lab: a French encoder-only model, much smaller than the 7B LLM, with a classification
head on top, **fully fine-tuned** (all weights trained) on the whole training set. All choices are made on
**`rand-dev`**, changing **one factor at a time**:

1. with vs without the lemma in the input (`camembert-base`);
2. `camembert-base` (110M parameters) vs `camembert-large` (335M).

The best configuration is run once on `rand-test` and `freq-test`, then compared with ICL and LoRA on the same test
items. Runtime: GPU (T4), a few minutes per training for `camembert-base`. On Kaggle, use *Run All* in an interactive
session.

In [1]:
# Setup: on Colab or Kaggle, clone (or update) the repository so that `src/` is importable; locally, run from the repo root.
import os, sys
REPO_URL = "https://github.com/Alhassliebe/LAB1_classification_with_llms.git"
ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
if "google.colab" in sys.modules or ON_KAGGLE:
    REPO_DIR = ("/kaggle/working" if ON_KAGGLE else "/content") + "/LAB1_classification_with_llms"
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull -q
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [2]:
# All hyperparameters of this notebook (usual values for fine-tuning BERT-like models; not tuned).
CONFIG = {
    "models": {"base": "almanach/camembert-base", "large": "almanach/camembert-large"},
    "learning_rate": {"base": 3e-5, "large": 1.5e-5},  # large encoders are less stable with a high learning rate
    "tuning_split": "rand-dev",
    "epochs": 4,
    "batch_size": 32,
    "warmup_share": 0.06,
    "weight_decay": 0.01,
    "max_length": 128,  # longer than 99 % of the inputs
    "eval_batch_size": 128,
    "seed": 42,
}

In [3]:
import gc
import time

# Kaggle offers two T4: with both, the Trainer would split each batch and double the batch size set in CONFIG.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import pandas as pd
import torch
import transformers
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer,
                          TrainingArguments)

from src.data import HYPERSENSES, SUPER2HYPER, SUPERSENSES, format_input, get_split, load_data
from src.evaluate import RESULTS_DIR, evaluate, log_experiment, paired_bootstrap, save_predictions

transformers.logging.set_verbosity_error()
GPU = torch.cuda.is_available()
df = load_data()
train = get_split(df, "train")

## Inputs

The input is the same text as for the LLMs (`Lemme : … / Définition : …`, or the definition alone); a linear layer
on the first token (`<s>`) maps it to the 25 supersenses. The hypersense is derived from the predicted supersense.

In [4]:
def encode(tokenizer, senses, with_lemma):
    """Tokenized inputs of `senses`, with their supersense index as label."""
    texts = [format_input(row, with_lemma) for _, row in senses.iterrows()]
    enc = tokenizer(texts, truncation=True, max_length=CONFIG["max_length"])
    labels = senses["supersense"].map(SUPERSENSES.index).tolist()
    return [{"input_ids": ids, "attention_mask": mask, "labels": label}
            for ids, mask, label in zip(enc["input_ids"], enc["attention_mask"], labels)]

## Training and prediction

Each configuration is trained once per session (a few minutes) and kept in memory for the test sets; prediction
files already in `results/predictions/` are reused instead of recomputed.

In [5]:
trained = {}


def get_model(size, with_lemma):
    """Fine-tuned model and tokenizer of one configuration, trained now if not already in this session."""
    name = config_id(size, with_lemma)
    device = "cuda" if GPU else "cpu"
    for other, (model, _) in trained.items():  # keep one model on the GPU, the others in CPU memory
        model.to(device if other == name else "cpu")
    gc.collect()
    torch.cuda.empty_cache()
    if name in trained:  # the test sets reuse the model trained for rand-dev
        return trained[name]
    transformers.set_seed(CONFIG["seed"])  # the classification head is initialised randomly
    tokenizer = AutoTokenizer.from_pretrained(CONFIG["models"][size])
    model = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["models"][size], num_labels=len(SUPERSENSES),
        id2label=dict(enumerate(SUPERSENSES)), label2id={s: i for i, s in enumerate(SUPERSENSES)})
    examples = encode(tokenizer, train, with_lemma)
    steps = CONFIG["epochs"] * -(-len(examples) // CONFIG["batch_size"])
    args = TrainingArguments(
        output_dir="outputs/camembert", num_train_epochs=CONFIG["epochs"],
        per_device_train_batch_size=CONFIG["batch_size"], learning_rate=CONFIG["learning_rate"][size],
        weight_decay=CONFIG["weight_decay"], lr_scheduler_type="linear",
        warmup_steps=int(CONFIG["warmup_share"] * steps), fp16=GPU, logging_steps=100, save_strategy="no",
        report_to="none", seed=CONFIG["seed"],
    )
    Trainer(model=model, args=args, train_dataset=examples,
            data_collator=DataCollatorWithPadding(tokenizer)).train()
    trained[name] = (model.eval(), tokenizer)
    return trained[name]


@torch.no_grad()
def predict(model, tokenizer, senses, with_lemma):
    """Predicted supersense of each sense, and the 3 most probable labels with their log-probabilities."""
    examples = encode(tokenizer, senses, with_lemma)
    collate = DataCollatorWithPadding(tokenizer)
    preds, raws = [], []
    for start in range(0, len(examples), CONFIG["eval_batch_size"]):
        batch = collate([{k: v for k, v in ex.items() if k != "labels"}
                         for ex in examples[start:start + CONFIG["eval_batch_size"]]]).to(model.device)
        top = model(**batch).logits.float().log_softmax(-1).topk(3)
        preds += [SUPERSENSES[i] for i in top.indices[:, 0].tolist()]
        raws += [" | ".join(f"{SUPERSENSES[i]} {s:.2f}" for s, i in zip(row_s, row_i))
                 for row_s, row_i in zip(top.values.tolist(), top.indices.tolist())]
    return preds, raws


def config_id(size, with_lemma):
    """Name of a configuration, e.g. base_lemma."""
    return f"{size}_{'lemma' if with_lemma else 'nolemma'}"


def run(size="base", with_lemma=True, split=CONFIG["tuning_split"]):
    """Train (or reuse) one configuration, predict `split`, save and log; reuse saved predictions if already run."""
    cid = config_id(size, with_lemma)
    path = RESULTS_DIR / "predictions" / f"camembert__{cid}__{split}.csv"
    fresh = not path.exists()
    if fresh:
        trained_now = cid not in trained
        start = time.time()
        model, tokenizer = get_model(size, with_lemma)
        eval_df = get_split(df, split)
        pred_labels, raws = predict(model, tokenizer, eval_df, with_lemma)
        runtime = (time.time() - start) / 60
        preds = eval_df[["sense_id", "lemma", "definition"]].assign(
            gold_supersense=eval_df["supersense"], pred_supersense=pred_labels,
            gold_hypersense=eval_df["hypersense"], raw_output=raws)
        preds.insert(6, "pred_hypersense", preds["pred_supersense"].map(SUPER2HYPER))
        save_predictions(preds, "camembert", cid, split)
    else:  # sessions are cut regularly: never pay twice for the same run
        preds = pd.read_csv(path, keep_default_na=False).replace("", None)
    sup = evaluate(preds["gold_supersense"].tolist(), preds["pred_supersense"].tolist(), SUPERSENSES)
    hyp = evaluate(preds["gold_hypersense"].tolist(), preds["pred_hypersense"].tolist(), HYPERSENSES)
    if fresh:
        log_experiment({"author": "A", "method": "camembert", "config_id": cid, "split": split,
                        "model": CONFIG["models"][size], "with_lemma": with_lemma, "head": "classification",
                        "decoding": "argmax", "epochs": CONFIG["epochs"],
                        "accuracy_super": round(sup["accuracy"], 4), "macro_f1_super": round(sup["macro_f1"], 4),
                        "accuracy_hyper": round(hyp["accuracy"], 4), "macro_f1_hyper": round(hyp["macro_f1"], 4),
                        "invalid_rate": round(sup["invalid_rate"], 4), "n": sup["n"], "runtime_min": round(runtime, 1),
                        "notes": "runtime includes training" if trained_now else ""})
    print(f"{cid:15s} macro-F1 {sup['macro_f1']:.3f}  acc {sup['accuracy']:.3f}  hyper macro-F1 {hyp['macro_f1']:.3f}")
    return {"config_id": cid, "macro_f1": sup["macro_f1"], "preds": preds}


def compare(a, b):
    """Print the share of bootstrap resamples where b has a higher macro-F1 than a.

    Close to 100 % or 0 % means a consistent difference, around 50 % means none.
    """
    share = paired_bootstrap(a["preds"]["gold_supersense"], a["preds"]["pred_supersense"],
                             b["preds"]["pred_supersense"], SUPERSENSES)
    print(f"{b['config_id']} beats {a['config_id']} in {share:.0%} of bootstrap resamples")

Each section keeps the setting with the higher macro-F1; a tie goes to the first, simpler one (with lemma, base).

## 1. Is the lemma useful?

In [6]:
lemma_runs = {True: run("base", True), False: run("base", False)}
compare(lemma_runs[False], lemma_runs[True])
WITH_LEMMA = lemma_runs[True]["macro_f1"] >= lemma_runs[False]["macro_f1"]
print("Lemma kept:", WITH_LEMMA)

base_lemma      macro-F1 0.506  acc 0.749  hyper macro-F1 0.511
base_nolemma    macro-F1 0.483  acc 0.733  hyper macro-F1 0.496
base_lemma beats base_nolemma in 89% of bootstrap resamples
Lemma kept: True


## 2. camembert-base vs camembert-large

In [7]:
size_runs = {"base": lemma_runs[WITH_LEMMA], "large": run("large", WITH_LEMMA)}
compare(size_runs["base"], size_runs["large"])
SIZE = max(size_runs, key=lambda s: size_runs[s]["macro_f1"])
print("Model kept:", SIZE)

large_lemma     macro-F1 0.640  acc 0.782  hyper macro-F1 0.639
large_lemma beats base_lemma in 100% of bootstrap resamples
Model kept: large


## Summary of the tuning runs

In [8]:
metrics = pd.read_csv(RESULTS_DIR / "metrics.csv")
metrics[(metrics["method"] == "camembert") & (metrics["split"] == CONFIG["tuning_split"])].drop_duplicates(
    "config_id", keep="last").sort_values("macro_f1_super", ascending=False)[
    ["config_id", "macro_f1_super", "accuracy_super", "macro_f1_hyper", "runtime_min"]]

,config_id,macro_f1_super,accuracy_super,macro_f1_hyper,runtime_min
21,large_lemma,0.6398,0.7825,0.6389,20.4
19,base_lemma,0.5061,0.7491,0.5111,6.3
20,base_nolemma,0.4828,0.7333,0.4962,5.9


## Final evaluation on the test sets

The configuration chosen on `rand-dev` above is run **once** on each test set; nothing is tuned after this point.

In [9]:
test_runs = {split: run(SIZE, WITH_LEMMA, split=split) for split in ["rand-test", "freq-test"]}

large_lemma     macro-F1 0.705  acc 0.807  hyper macro-F1 0.677
large_lemma     macro-F1 0.639  acc 0.682  hyper macro-F1 0.651


## Comparison with ICL and LoRA

Final configuration of each method on the same test items; bootstrap: share of resamples where CamemBERT has a
higher macro-F1 than the other method.

In [10]:
FINAL = {"icl": "en_5shot_similar_nolemma_logprob", "lora": "all_generation_lemma"}
rows = []
for split, cam in test_runs.items():
    for method, cid in FINAL.items():
        other = pd.read_csv(RESULTS_DIR / "predictions" / f"{method}__{cid}__{split}.csv", keep_default_na=False)
        both = other.merge(cam["preds"][["sense_id", "pred_supersense"]], on="sense_id", suffixes=("", "_cam"))
        sup = evaluate(both["gold_supersense"].tolist(), both["pred_supersense"].tolist(), SUPERSENSES)
        share = paired_bootstrap(both["gold_supersense"], both["pred_supersense"], both["pred_supersense_cam"],
                                 SUPERSENSES)
        rows.append({"split": split, "method": method, "macro_f1": round(sup["macro_f1"], 3),
                     "accuracy": round(sup["accuracy"], 3), "CamemBERT better (bootstrap)": f"{share:.0%}"})
    sup = evaluate(cam["preds"]["gold_supersense"].tolist(), cam["preds"]["pred_supersense"].tolist(), SUPERSENSES)
    rows.append({"split": split, "method": "camembert", "macro_f1": round(sup["macro_f1"], 3),
                 "accuracy": round(sup["accuracy"], 3), "CamemBERT better (bootstrap)": ""})
pd.DataFrame(rows)

,split,method,macro_f1,accuracy,CamemBERT better (bootstrap)
0,rand-test,icl,0.583,0.717,100%
1,rand-test,lora,0.719,0.820,29%
2,rand-test,camembert,0.705,0.807,
3,freq-test,icl,0.519,0.523,100%
4,freq-test,lora,0.675,0.702,0%
5,freq-test,camembert,0.639,0.682,


## Conclusions

- The lemma helps CamemBERT a little: macro-F1 0.506 with it vs 0.483 without on `rand-dev` (89 % of bootstrap
  resamples), the same direction as for LoRA and the opposite of ICL.
- `camembert-large` is much better than `camembert-base`: 0.640 vs 0.506 (100 % of resamples), for 20 instead of
  6 minutes of training on a T4. Final configuration: `camembert-large` with the lemma (`large_lemma`).
- On the test sets, CamemBERT-large is far better than ICL (macro-F1 0.705 vs 0.583 on `rand-test`, 0.639 vs 0.519 on
  `freq-test`, 100 % of resamples). It cannot be distinguished from LoRA on `rand-test` (0.705 vs 0.719, CamemBERT
  better in 29 % of resamples) and is worse on `freq-test` (0.639 vs 0.675, 0 %).
- A 335M-parameter encoder trained in 20 minutes is close to a 7B LLM fine-tuned with QLoRA in about an hour, and
  predicts a test set in seconds. The gap on `freq-test` (frequent, polysemous lemmas) suggests that the LLM's prior
  knowledge of the words helps most where the definitions are hardest.